# Surrogate analysis

- **Consensus fidelity and final-state MCC are out-of-sample.** Runs are split into 5 folds (by run,
  stratified by graph type); each surrogate is fitted on 4 folds and scored on the held-out one, so every
  run is scored once by a model that never saw it. Consensus fidelity comes from stochastic rollouts from
  round 0; final-state MCC from one-step predictions of each agent's final belief. CIs bootstrap over the
  16 network realizations per family (graph type × seed, 20 runs each).
- **AIC, BIC and pseudo-R² are in-sample**: each surrogate is fitted once on all 320 runs.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

from exp_utils import REPO_ROOT, save_table
from exp_plot_utils import (
    _configure_fonts,
    OKABE_ITO,
    GRAPH_COLOR,
    GRAPH_MAP,
    SCENARIO_MAP,
    SCENARIO_EXTENDED_MAP,
    SCENARIO_ORDER,
)

SURROGATE_SUMMARY = (
    REPO_ROOT / "data" / "outputs" / "surrogates" / "evaluation" / "summary.csv"
)
FIG_DIR = (
    REPO_ROOT / "data" / "analysis" / "aggregated_expanded" / "surrogate" / "figures"
)
FIG_DIR.mkdir(parents=True, exist_ok=True)

SURROGATE_MAP = {"m1": "M1", "m2-5": "M2", "m3": "M3", "m4": "M4"}
GRAPH_ORDER = ["erdos-renyi", "watts-strogatz"]
NETWORK_SHORT = {"erdos-renyi": "ER", "watts-strogatz": "WS"}

## 1. Load the surrogate evaluation

One row per model × scenario × graph family: cross-validated behaviour metrics with 95% network-bootstrap
CIs, and in-sample information criteria.

In [ ]:
METRICS = {"mcc": "final_state_mcc", "consensus": "consensus_fidelity"}

summary = pd.read_csv(SURROGATE_SUMMARY)
surrogate_df = summary[
    [
        "model",
        "scenario",
        "graph_family",
        "n_runs",
        "n_runs_new",
        "n_agents",
        "n_params",
        "log_lik",
        "log_lik_null",
        "pseudo_r2",
        "aic",
        "bic",
        "delta_aic",
        "delta_bic",
    ]
].assign(
    **{
        f"{short}_{stat}": summary[f"{col}__{stat}"]
        for short, col in METRICS.items()
        for stat in ["point", "ci_lower", "ci_upper"]
    }
)

expected = len(SURROGATE_MAP) * len(SCENARIO_ORDER) * len(GRAPH_ORDER)
assert (
    len(surrogate_df) == expected
    and not surrogate_df.duplicated(["model", "scenario", "graph_family"]).any()
), f"{len(surrogate_df)} / {expected} model x scenario x family cells"

save_table(
    surrogate_df, "E4_surrogate_plot_data"
)  # data behind every figure in this notebook
surrogate_df.sort_values(["scenario", "model", "graph_family"]).round(3)

## 2. Figures

In [ ]:
_configure_fonts()
sns.set_theme(style="whitegrid", context="talk")

MODEL_ORDER = list(SURROGATE_MAP)
x = np.arange(len(MODEL_ORDER))
OFFSETS = {"erdos-renyi": -0.07, "watts-strogatz": 0.07}
PANEL_LABELS = ["A", "B", "C", "D"]
GRAPH_HANDLES = [
    Line2D(
        [0],
        [0],
        color=GRAPH_COLOR[g],
        marker="o",
        linestyle="-",
        lw=2.2,
        ms=7,
        label=GRAPH_MAP[g],
    )
    for g in GRAPH_ORDER
]


def _ylim(df, lo_col, hi_col):
    lo, hi = df[lo_col].min(), df[hi_col].max()
    pad = max(0.02, 0.08 * (hi - lo))
    return max(-1.0, lo - pad), min(1.0, hi + pad)


def _style_axis(ax):
    ax.set_xticks(x)
    ax.set_xticklabels([SURROGATE_MAP[m] for m in MODEL_ORDER])
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.25, linestyle="--", linewidth=1)
    ax.grid(axis="x", visible=False)
    for xv in x:
        ax.axvline(
            x=xv,
            color=OKABE_ITO["black"],
            linestyle=(0, (1.0, 3.0)),
            linewidth=1.0,
            alpha=0.40,
            dash_capstyle="round",
            zorder=0,
        )
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(OKABE_ITO["black"])
        ax.spines[side].set_linewidth(1.5)
    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        bottom=True,
        left=True,
        width=1.3,
        length=6,
        color=OKABE_ITO["black"],
        labelcolor=OKABE_ITO["black"],
        labelsize=12,
    )


def plot_2x2(df, metric, ylabel, save_path):
    """One panel per scenario; surrogates on x, one series per graph family, 95% CIs."""
    ylim = _ylim(df, f"{metric}_ci_lower", f"{metric}_ci_upper")
    fig, axes = plt.subplots(2, 2, figsize=(6.5, 6.5))
    for i, scenario in enumerate(SCENARIO_ORDER):
        ax = axes[i // 2, i % 2]
        panel = df[df["scenario"] == scenario]
        for graph in GRAPH_ORDER:
            sub = (
                panel[panel["graph_family"] == graph]
                .set_index("model")
                .reindex(MODEL_ORDER)
            )
            y = sub[f"{metric}_point"].to_numpy(dtype=float)
            # Clip at 0: a point estimate can fall outside its percentile-bootstrap CI.
            yerr = np.clip(
                [
                    y - sub[f"{metric}_ci_lower"].to_numpy(dtype=float),
                    sub[f"{metric}_ci_upper"].to_numpy(dtype=float) - y,
                ],
                0,
                None,
            )
            ax.errorbar(
                x + OFFSETS[graph],
                y,
                yerr=yerr,
                fmt="o-",
                color=GRAPH_COLOR[graph],
                lw=2.0,
                ms=4,
                capsize=4,
                alpha=0.95,
            )
        ax.set_title(SCENARIO_MAP[scenario])
        _style_axis(ax)
        ax.text(
            -0.24 if i % 2 else -0.30,
            1.10,
            PANEL_LABELS[i],
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=14,
            fontweight="bold",
            color=OKABE_ITO["black"],
        )
        ax.set_ylim(ylim)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    for row in range(2):
        axes[row, 0].set_ylabel(ylabel)

    fig.legend(
        handles=GRAPH_HANDLES,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.03),
        ncol=2,
        frameon=False,
    )
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.14, wspace=0.3)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)


plot_2x2(
    surrogate_df, "consensus", "Consensus Fidelity", FIG_DIR / "consensus-fidelity.pdf"
)
plot_2x2(surrogate_df, "mcc", "Final-state MCC", FIG_DIR / "final-state-mcc.pdf")


## 3. LaTeX tables

In [ ]:
def surrogate_table(df, metric, label):
    """Scenario x surrogate x network table in percent, with 95% CI bounds."""
    out = pd.DataFrame(
        {
            "Scenario": pd.Categorical(
                df["scenario"].map(SCENARIO_EXTENDED_MAP),
                categories=[SCENARIO_EXTENDED_MAP[s] for s in SCENARIO_ORDER],
                ordered=True,
            ),
            "Surrogate": pd.Categorical(
                df["model"].map(SURROGATE_MAP),
                categories=list(SURROGATE_MAP.values()),
                ordered=True,
            ),
            "Network": pd.Categorical(
                df["graph_family"].map(NETWORK_SHORT),
                categories=list(NETWORK_SHORT.values()),
                ordered=True,
            ),
            label: 100 * df[f"{metric}_point"],
            "CI low": 100 * df[f"{metric}_ci_lower"],
            "CI high": 100 * df[f"{metric}_ci_upper"],
        }
    )
    return out.set_index(["Scenario", "Surrogate", "Network"]).sort_index().round(1)


def to_latex(table, label):
    table = table.rename(
        columns=lambda c: c.replace("%", r"\%")
    )  # bare % starts a LaTeX comment
    latex = table.to_latex(
        index=True,
        caption="",
        multicolumn=True,
        multirow=True,
        escape=False,
        float_format="{:0.1f}".format,
        label=label,
    )
    return latex.replace(
        "\\multirow[t]{", "\\multirow[c]{"
    )  # vertically centre multirow labels


SURROGATE_LEGEND = (
    r"Surrogate models are M1 (persistence), M2 (persistence + global belief composition), "
    r"M3 (persistence + local neighborhood composition), and M4 (M3 + agent identity and social role); "
    r"see~\Cref{tab:surrogate_models}. "
)
NETWORK_NOTE = r"In the network column, ER stands for Erd\H{o}s--R\'enyi, and WS stands for Watts--Strogatz."
TABLE_CAPTIONS = {
    "consensus": (
        r"\textbf{Consensus fidelity by scenario, surrogate model, and network structure}~(\Cref{app:dynamics}) "
        r"with \(95\%\) confidence intervals from resampling the 16 network realizations. Consensus fidelity is the similarity between observed and "
        r"surrogate final-round consensus levels across runs (out-of-sample, 5-fold cross-validation); higher values "
        r"indicate that a surrogate more accurately reproduces the population-level agreement observed in "
        r"\texttt{CoevolveSim}. " + SURROGATE_LEGEND + NETWORK_NOTE
    ),
    "mcc": (
        r"\textbf{Final-state Matthews Correlation Coefficient (final-state MCC) by scenario, surrogate model, and "
        r"network structure}~(\Cref{app:dynamics}). We report MCC with \(95\%\) confidence intervals (resampling the 16 network realizations) for "
        r"agreement between each surrogate's predicted final belief state and the observed final belief state "
        r"(out-of-sample, 5-fold cross-validation). Higher final-state MCC indicates more accurate prediction of "
        r"individual final-round beliefs. " + SURROGATE_LEGEND + NETWORK_NOTE
    ),
}
PAPER_HEADER = {
    "consensus": r"\makecell{Consensus \\ fidelity (\%)}",
    "mcc": r"\makecell{Final-state \\ MCC (\%)}",
}


def to_paper_latex(table, metric, label):
    """Paper-ready float: centred, column-width, caption below, makecell headers."""
    table = table.rename(
        columns={
            table.columns[0]: PAPER_HEADER[metric],
            "CI low": r"\makecell{\(95\%\) CI \\ lower}",
            "CI high": r"\makecell{\(95\%\) CI \\ upper}",
        }
    )
    body = table.to_latex(
        index=True,
        multicolumn=True,
        multirow=True,
        escape=False,
        column_format="lllccc",
        float_format="{:0.1f}".format,
    )
    body = body.replace("\\multirow[t]{", "\\multirow[c]{")
    return "\n".join(
        [
            r"\begin{table}[ht]",
            r"\centering",
            r"\begin{adjustbox}{width=\columnwidth}",
            body.strip(),
            r"\end{adjustbox}",
            rf"\caption{{{TABLE_CAPTIONS[metric]}}}",
            rf"\label{{{label}}}",
            r"\end{table}",
        ]
    )


for metric, label, stem, tex_label in [
    (
        "consensus",
        "Consensus fidelity (%)",
        "consensus_fidelity",
        "tab:consensus-fidelity",
    ),
    ("mcc", "Final-state MCC (%)", "final_state_mcc", "tab:final-mcc"),
]:
    table = surrogate_table(surrogate_df, metric, label)
    latex = to_paper_latex(table, metric, tex_label)
    tex_path = save_table(table, f"E4_{stem}", index=True).with_suffix(".tex")
    tex_path.write_text(latex + "\n")
    print("Saved:", tex_path.relative_to(REPO_ROOT))
    print(latex)
